# Lab 5: Neural Networks – Analysis Tasks

**Task 1:** Determine whether the `LinearRegressionNN` model is capable of learning from each dataset provided in Lab 4.

**Task 2:** Determine whether `LinearRegressionNN` is suitable for `binary_classification_moons.csv`, evaluate `ModelV0`, and propose an improved model.

## Task 1: Suitability of `LinearRegressionNN` for Lab 4 Datasets

### Dataset: `linear-regression-data1.csv`
- **Data shape:** 51 samples, 1 input feature `x` and target `y`.
- **Pattern:** Strictly linear relationship: `y ≈ 3x + 0.3`.
- **Model capacity:** `LinearRegressionNN` contains a single linear layer `nn.Linear(1, 1)`, which is exactly the right capacity for a linear mapping from 1-D input to 1-D output.
- **Verdict:** ✅ **Suitable.** The model is appropriate.

---

### Dataset: `assignment-data.csv`
- **Data shape:** 51 samples, 1 input feature `x` and target `y`.
- **Pattern:** Quadratic relationship. For example:
  - x=0 → y=2.00
  - x=1 → y=5.00
  - x=2 → y=14.00
  - x=3 → y=29.00
  The second differences are constant (+6), confirming `y` grows quadratically with `x`.
- **Model capacity:** A single linear layer can only represent lines (`y = wx + b`), not parabolas.
- **Verdict:** ❌ **Not suitable.** The model architecture is too simple for this nonlinear dataset.

---

### Dataset: `assignment-data2.csv`
- **Data shape:** 100 samples, 1 input feature `x` and target `y`.
- **Pattern:** Quadratic, symmetric parabola with minimum near x=0 (e.g., x=0 → y=1.00; x=1 → y=8.00; x=-1 → y=8.00). Fits `y ≈ 1 + 7x²`.
- **Model capacity:** Same as above — single linear layer cannot capture quadratic curves.
- **Verdict:** ❌ **Not suitable.** The model architecture is too simple for this nonlinear dataset.

## Verification: Trend Checks

In [ ]:
import pandas as pd

df1 = pd.read_csv('linear-regression-data1.csv')
df2 = pd.read_csv('assignment-data.csv')
df3 = pd.read_csv('assignment-data2.csv')

print('=== linear-regression-data1 ===')
print(df1.head(10))
print('\n=== assignment-data (first and last rows) ===')
print(df2.head(5))
print(df2.tail(5))
print('\n=== assignment-data2 (middle, near minimum) ===')
print(df3.iloc[45:55])

## Task 2: Binary Classification with `binary_classification_moons.csv`

### 2.1 – Is `LinearRegressionNN` suitable?
- `LinearRegressionNN` (from Task 1) has a single linear layer `nn.Linear(1,1)`.
- It outputs unbounded real-valued scores, not class probabilities.
- It lacks nonlinearity, so the decision boundary is always a straight line (or hyperplane).
- The `moons` dataset is nonlinear and requires a curved/nonlinear decision boundary.
- **Verdict:** ❌ **Not suitable.**

---

### 2.2 – Is `ModelV0` suitable?

```python
class ModelV0(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=2, out_features=5)
        self.layer_2 = nn.Linear(in_features=5, out_features=1)

    def forward(self, x):
        return self.layer_2(self.layer_1(x))
```

- **Architecture:** A 2-layer MLP.
- **Missing nonlinearity:** No activation function. Composing two linear layers is mathematically equivalent to a single linear layer (`y = W2(W1x + b1) + b2` → linear in x).
- **Hidden units:** 5 hidden units are very few to represent the complex interleaved half-moon shapes.
- **Output:** Raw logit without a sigmoid/nonlinearity applied at the end.
- **Verdict:** ❌ **Not suitable.** The model is effectively linear and will not separate the two moon-shaped classes.

## Task 3: Improved Model for `binary_classification_moons.csv`

To build a model suitable for the `moons` dataset, the following changes are required:
1. **Nonlinear activations** (ReLU) after hidden layers to allow the network to model nonlinear boundaries.
2. **Sigmoid activation** on the final output to produce class probabilities.
3. **More hidden units** (e.g., 16 or 32) to capture the curved structure.
4. Optionally, **more hidden layers** for greater representational capacity.

Below is a definition of an improved model (`ModelV1`) and a brief evaluation of its suitability.

In [ ]:
import torch
import torch.nn as nn

class ModelV1(nn.Module):
    def __init__(self, hidden_units=16):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Linear(in_features=2, out_features=hidden_units),
            nn.ReLU(),
            nn.Linear(in_features=hidden_units, out_features=hidden_units),
            nn.ReLU(),
            nn.Linear(in_features=hidden_units, out_features=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.layer_stack(x)

# Instantiate and inspect
model = ModelV1(hidden_units=16)
print(model)

# Quick forward-pass sanity check with dummy input (batch_size=4, features=2)
dummy_input = torch.randn(4, 2)
output = model(dummy_input)
print('\nDummy output shape:', output.shape)
print('Dummy output:', output.detach().numpy())

### Why `ModelV1` is suitable:

- **ReLU activations** introduce nonlinearity, enabling the model to approximate the complex nonlinear decision boundary between the two moon classes.
- **Two hidden layers** with 16 hidden units each provide enough capacity to model the curved shape without overfitting on small toy datasets.
- **Sigmoid output** maps the final logit to `[0, 1]`, giving a valid probability for binary classification.
- This architecture is a standard, proven starting point for small binary classification problems like the `moons` dataset.

In summary, replacing `ModelV0`'s linear stack with `ModelV1`'s nonlinear, wider, deeper architecture makes it well-suited for `binary_classification_moons.csv`.